# Cache NeuroMorpho neurons, then compare them with `braincell`

The notebook has two steps:

1. Optional, needs network access: use a typed `NeuroMorphoQuery` to search NeuroMorpho and cache every match under `data/neuromorpho/<neuron_id>/` via `NeuroMorphoClient.download`. Each folder holds the standardized `SWC` and a `metadata.json` with the neuron metadata, the published NeuroMorpho measurement and the download links.
2. Scan a cache with `NeuroMorphoCache`, load each standardized `SWC` and compare `braincell`'s metrics with the cached measurement.

By default (`DOWNLOAD = False`) the notebook runs offline on the fixed dataset in `data/morphology/neuromorpho/`: seven NeuroMorpho neurons, redistributed under CC BY 4.0 (sources, hashes and the required citations are in that folder's `README.md`). The offline regression test `braincell/io/neuromorpho/regression_test.py` makes the same comparison, with the same formulas, thresholds and known exceptions. Set `DOWNLOAD = True` to query, download and compare your own selection instead.

For one-shot use cases that just need a `Morphology` object, prefer the Tier-1 helper:

```python
from braincell.io import load_neuromorpho

morph = load_neuromorpho(10047)              # one call → parsed Morphology
morph = Morphology.from_neuromorpho(10047)   # equivalent classmethod entry point
```

This notebook uses the Tier-2 client (`NeuroMorphoClient`) and the cache object (`NeuroMorphoCache`) so it can iterate over many neurons and validate `braincell`'s metric computations against the upstream measurement records.

In [1]:
import math
import os
import sys
from pathlib import Path

import brainunit as u
import numpy as np

os.environ.setdefault("JAX_PLATFORMS", "cpu")


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "examples").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import braincell
from braincell import Morphology
from braincell.io import NeuroMorphoCache, NeuroMorphoClient, NeuroMorphoQuery, load_neuromorpho

# False: compare the fixed dataset offline. True: query NeuroMorpho and download (network access).
DOWNLOAD = False
if DOWNLOAD:
    output_root = repo_root / "data" / "neuromorpho"
else:
    output_root = repo_root / "data" / "morphology" / "neuromorpho"

print("braincell version:", braincell.__version__)
print("DOWNLOAD:", DOWNLOAD)
print("output_root:", output_root.relative_to(repo_root))

braincell version: 0.2.0
DOWNLOAD: False
output_root: data/morphology/neuromorpho


In [2]:
# Formulas, thresholds and known exceptions of braincell/io/neuromorpho/regression_test.py.
EXACT_MATCH_KEYS = ("n_stems", "n_branch", "n_bifs")
FLOAT_THRESHOLDS = {
    "length": 1e-4,
    "pathDistance": 1e-4,
    "eucDistance": 1e-4,
    "surface": 1e-4,
    "volume": 1e-4,
    "soma_Surface": 2e-2,
}
REQUIRED_MEASUREMENT_KEYS = EXACT_MATCH_KEYS + tuple(FLOAT_THRESHOLDS)

# (neuron id, metric) -> reason, for the fixed dataset. A metric listed here that
# fails is reported as "registered" instead of "FAIL".
MULTI_POINT_SOMA = "multi-point soma: soma branches count as stems; distances start at the soma boundary"
CON2PROX = "con2prox: L-Measure counts a bifurcation where the neuromorpho mode attaches at x = 0"
KNOWN_EXCEPTIONS = {
    (100, "n_stems"): MULTI_POINT_SOMA,
    (100, "pathDistance"): MULTI_POINT_SOMA,
    (100, "eucDistance"): MULTI_POINT_SOMA,
    (100, "soma_Surface"): MULTI_POINT_SOMA,
    (1013, "n_stems"): MULTI_POINT_SOMA,
    (1013, "n_branch"): CON2PROX,
    (1013, "n_bifs"): CON2PROX,
    (1013, "pathDistance"): MULTI_POINT_SOMA,
    (1013, "soma_Surface"): MULTI_POINT_SOMA,
    (10036, "n_branch"): CON2PROX,
    (10036, "n_bifs"): CON2PROX,
}


def missing_measurement_keys(measurement):
    return [key for key in REQUIRED_MEASUREMENT_KEYS if measurement.get(key) is None]


def has_soma_branch(tree):
    return any(branch.type == "soma" for branch in tree.branches)


def compute_braincell_metrics(tree):
    soma_branches = [branch for branch in tree.branches if branch.type == "soma"]
    non_soma_branches = [branch for branch in tree.branches if branch.type != "soma"]
    # NeuroMorpho (L-Measure) sums cylinders with the child point's radius, so
    # surface and volume use each segment's distal radius. Branch.area and
    # Branch.volume are frustums and differ on tapered dendrites.
    surface = volume = 0.0
    for branch in non_soma_branches:
        lengths = np.asarray(branch.branch.lengths.to_decimal(u.um))
        radii = np.asarray(branch.branch.radii_distal.to_decimal(u.um))
        surface += float(np.sum(2.0 * math.pi * radii * lengths))
        volume += float(np.sum(math.pi * radii * radii * lengths))
    return {
        "n_stems": int(tree.n_stems),
        "n_branch": int(len(non_soma_branches)),
        "n_bifs": int(sum(branch.n_children >= 2 for branch in non_soma_branches)),
        "length": float(sum(branch.length.to_decimal(u.um) for branch in non_soma_branches)),
        "surface": surface,
        "volume": volume,
        "pathDistance": float(tree.max_path_distance_excluding_soma.to_decimal(u.um)),
        "eucDistance": float(tree.max_euclidean_distance_excluding_soma.to_decimal(u.um)),
        "soma_Surface": float(sum(branch.area.to_decimal(u.um ** 2) for branch in soma_branches)),
    }


def compute_neuromorpho_metrics(measurement):
    return {
        "n_stems": int(round(float(measurement["n_stems"]))),
        "n_branch": int(round(float(measurement["n_branch"]))),
        "n_bifs": int(round(float(measurement["n_bifs"]))),
        "length": float(measurement["length"]),
        "surface": float(measurement["surface"]),
        "volume": float(measurement["volume"]),
        "pathDistance": float(measurement["pathDistance"]),
        "eucDistance": float(measurement["eucDistance"]),
        "soma_Surface": float(measurement["soma_Surface"]),
    }


def compare_tree_with_measurement(tree, measurement, neuron_id):
    braincell_metrics = compute_braincell_metrics(tree)
    neuromorpho_metrics = compute_neuromorpho_metrics(measurement)
    metric_results = {}

    for key, target in neuromorpho_metrics.items():
        actual = braincell_metrics[key]
        if key in EXACT_MATCH_KEYS:
            rel_diff = None
            passed = actual == target
        else:
            rel_diff = abs(actual - target) / abs(target)
            passed = rel_diff <= FLOAT_THRESHOLDS[key]
        if passed:
            status = "pass"
        elif (neuron_id, key) in KNOWN_EXCEPTIONS:
            status = "registered"
        else:
            status = "FAIL"
        metric_results[key] = {
            "braincell": actual,
            "neuromorpho": target,
            "rel_diff": rel_diff,
            "threshold": FLOAT_THRESHOLDS.get(key),
            "status": status,
            "reason": KNOWN_EXCEPTIONS.get((neuron_id, key)) if status == "registered" else None,
        }

    statuses = {result["status"] for result in metric_results.values()}
    if "FAIL" in statuses:
        overall = "failed"
    elif "registered" in statuses:
        overall = "registered exceptions only"
    else:
        overall = "passed"
    return {"metric_results": metric_results, "overall": overall}

## 1. Batch query and cache neurons (optional)

These two cells run only with `DOWNLOAD = True`. Use a typed `NeuroMorphoQuery` (or a raw Solr `q`/`fq` string) and `client.iter_search(...)` to lazily stream every matching neuron, then download each into:

`data/neuromorpho/<neuron_id>/`

Each neuron folder will contain:

- the standardized `SWC`, the original file, or both
- `metadata.json`
- the full NeuroMorpho measurement payload and links needed to trace the neuron later

In [3]:
if not DOWNLOAD:
    print("DOWNLOAD is False: no query; the comparison below reads the fixed dataset.")
else:
    query = NeuroMorphoQuery(
        species="mouse",
        brain_region="cerebellum",
    )
    size = 100
    page_start = 5
    limit = 500  # set to None to stream every match
    sort = "neuron_id,asc"
    download_mode = "standard"  # choose from: "standard", "original", "both"
    overwrite = False

    client = NeuroMorphoClient(cache_dir=output_root)
    neurons = list(
        client.iter_search(
            query,
            size=size,
            start_page=page_start,
            limit=limit,
            sort=sort,
        )
    )
    if not neurons:
        raise RuntimeError(
            "No NeuroMorpho records matched the current query. Relax the filters and run again."
        )

    print("query.q:", query.to_q())
    print("query.fq:", query.to_fq())
    print("size:", size)
    print("page_start:", page_start)
    print("limit:", limit)
    print("download_mode:", download_mode)
    print("output_root:", output_root.relative_to(repo_root))
    print()
    print("matched_neurons:", len(neurons))
    print()
    for neuron in neurons[:10]:
        print(f"id={neuron.neuron_id} name={neuron.neuron_name}")
        print("  archive:", neuron.archive)
        print("  brain_region:", neuron.brain_region)
        print("  original_format:", neuron.original_format)
        print()
    if len(neurons) > 10:
        print(f"... and {len(neurons) - 10} more")

DOWNLOAD is False: no query; the comparison below reads the fixed dataset.


In [4]:
if not DOWNLOAD:
    print("DOWNLOAD is False: nothing to download.")
else:
    cache = NeuroMorphoCache(output_root)
    download_records = []
    for neuron in neurons:
        record = client.download(
            neuron,
            output_dir=output_root,
            mode=download_mode,
            overwrite=overwrite,
        )
        download_records.append(record)

        print(f"cached neuron_id={neuron.neuron_id} name={neuron.neuron_name}")
        print("  folder:", record.folder)
        print("  metadata:", record.metadata_path)
        for item in record.download_items:
            print(f"  {item.kind}: {item.filename}")
            print("    url:", item.url)
            print("    path:", item.path)
            print("    downloaded_now:", item.downloaded_now)
            print("    reason:", item.reason)
        print()

    if download_records:
        sample_neuron = neurons[0]
        sample_metadata = cache.metadata(sample_neuron.neuron_id)
        sample_detail = client.describe(sample_neuron)
        print("sample_metadata_file:", download_records[0].metadata_path)
        print("sample_measurement_url:", sample_metadata.get("measurement_url"))
        print("sample_thumbnail_url:", sample_detail.urls.thumbnail)
        print("sample_standard_swc_url:", sample_detail.urls.standard_swc)
        print("sample_original_file_url:", sample_detail.urls.original_file)
        print("sample_measurement (raw upstream payload):")
        print(sample_metadata["measurement"])
        print()
        print("typed measurement (snake_case attributes):")
        typed_meas = sample_detail.measurement
        print("  length:", typed_meas.length)
        print("  surface:", typed_meas.surface)
        print("  path_distance:", typed_meas.path_distance)
        print("  euclidean_distance:", typed_meas.euclidean_distance)
        print("  soma_surface:", typed_meas.soma_surface)

DOWNLOAD is False: nothing to download.


## 2. Scan cached folders and batch compare

This step does not query NeuroMorpho. It uses `NeuroMorphoCache` to enumerate the neurons cached under `output_root`, loads each standardized `SWC` and compares the resulting `Morphology` with the upstream `measurement` stored in `metadata.json`.

Counts must be equal; the other metrics must agree to the relative tolerances in `FLOAT_THRESHOLDS`. Surface and volume follow NeuroMorpho's L-Measure formulas (cylinders with each segment's distal radius), not `Branch.area` and `Branch.volume`. A failing metric listed in `KNOWN_EXCEPTIONS` is marked `registered`. On the fixed dataset four neurons pass every metric and three have registered exceptions only (multi-point somata and `con2prox` points, explained in the regression test).

In [5]:
cache = NeuroMorphoCache(output_root)
cached_ids = cache.list_neurons()

comparison_runs = []
summary = {
    "scanned": 0,
    "compared": 0,
    "skipped": 0,
    "passed": 0,
    "registered exceptions only": 0,
    "failed": 0,
    "errors": 0,
}

if not cached_ids:
    print("No cached neuron folders were found under", output_root.relative_to(repo_root))
else:
    for neuron_id in cached_ids:
        summary["scanned"] += 1
        folder = cache.layout.neuron_dir(neuron_id)

        try:
            metadata = cache.metadata(neuron_id)
        except FileNotFoundError:
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: missing metadata.json")
            continue
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to read metadata.json: {exc}")
            continue

        measurement = metadata.get("measurement")
        if not measurement:
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: metadata.json does not contain measurement")
            continue

        missing_keys = missing_measurement_keys(measurement)
        if missing_keys:
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: incomplete measurement for keys {missing_keys}")
            continue

        swc_path = cache.standard_swc_path(neuron_id)
        if swc_path is None or not swc_path.exists():
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: no standardized SWC found")
            continue

        try:
            tree = Morphology.from_swc(swc_path, mode="neuromorpho")
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to load {swc_path.name}: {exc}")
            continue

        if not has_soma_branch(tree):
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: no soma branches in {swc_path.name}")
            continue

        try:
            comparison = compare_tree_with_measurement(tree, measurement, neuron_id)
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to compare {swc_path.name}: {exc}")
            continue

        summary["compared"] += 1
        summary[comparison["overall"]] += 1
        comparison_runs.append(
            {
                "neuron_id": metadata.get("neuron_id"),
                "neuron_name": metadata.get("neuron_name"),
                "swc_path": swc_path,
                "comparison": comparison,
            }
        )

        print("=" * 80)
        print(f"neuron_id: {metadata.get('neuron_id')}  name: {metadata.get('neuron_name')}  swc: {swc_path.name}")
        for key, result in comparison["metric_results"].items():
            rel_text = "" if result["rel_diff"] is None else f", rel_diff={result['rel_diff']:.2e} (<= {result['threshold']:g})"
            reason = f"  [{result['reason']}]" if result["reason"] else ""
            if key in EXACT_MATCH_KEYS:
                values = f"braincell={result['braincell']}, neuromorpho={result['neuromorpho']}"
            else:
                values = f"braincell={result['braincell']:.6f}, neuromorpho={result['neuromorpho']:.6f}"
            print(f"  {key}: {values}{rel_text}: {result['status']}{reason}")
        print(f"  overall: {comparison['overall']}")

    print("=" * 80)
    print("scan summary")
    for key, value in summary.items():
        print(f"{key}: {value}")

neuron_id: 100  name: n419  swc: n419.CNG.swc
  n_stems: braincell=4, neuromorpho=5: registered  [multi-point soma: soma branches count as stems; distances start at the soma boundary]
  n_branch: braincell=155, neuromorpho=155: pass
  n_bifs: braincell=75, neuromorpho=75: pass
  length: braincell=13142.972197, neuromorpho=13143.000000, rel_diff=2.12e-06 (<= 0.0001): pass
  surface: braincell=23909.319848, neuromorpho=23909.300000, rel_diff=8.30e-07 (<= 0.0001): pass
  volume: braincell=4215.038025, neuromorpho=4215.040000, rel_diff=4.69e-07 (<= 0.0001): pass
  pathDistance: braincell=1074.935488, neuromorpho=1078.420000, rel_diff=3.23e-03 (<= 0.0001): registered  [multi-point soma: soma branches count as stems; distances start at the soma boundary]
  eucDistance: braincell=526.257240, neuromorpho=529.358000, rel_diff=5.86e-03 (<= 0.0001): registered  [multi-point soma: soma branches count as stems; distances start at the soma boundary]
  soma_Surface: braincell=696.158987, neuromorpho=

neuron_id: 10074  name: Purkinje-slice-ageP35-3  swc: Purkinje-slice-ageP35-3.CNG.swc
  n_stems: braincell=1, neuromorpho=1: pass
  n_branch: braincell=605, neuromorpho=605: pass
  n_bifs: braincell=302, neuromorpho=302: pass
  length: braincell=5915.109829, neuromorpho=5915.110000, rel_diff=2.90e-08 (<= 0.0001): pass
  surface: braincell=32214.353370, neuromorpho=32214.400000, rel_diff=1.45e-06 (<= 0.0001): pass
  volume: braincell=15185.347856, neuromorpho=15185.300000, rel_diff=3.15e-06 (<= 0.0001): pass
  pathDistance: braincell=298.693124, neuromorpho=298.690000, rel_diff=1.05e-05 (<= 0.0001): pass
  eucDistance: braincell=246.927589, neuromorpho=246.920000, rel_diff=3.07e-05 (<= 0.0001): pass
  soma_Surface: braincell=180.133308, neuromorpho=180.080000, rel_diff=2.96e-04 (<= 0.02): pass
  overall: passed
neuron_id: 102085  name: 01042017e6-000  swc: 01042017e6-000.CNG.swc
  n_stems: braincell=1, neuromorpho=1: pass
  n_branch: braincell=15, neuromorpho=15: pass
  n_bifs: braincel

neuron_id: 122127  name: AA0990  swc: AA0990.CNG.swc
  n_stems: braincell=2, neuromorpho=2: pass
  n_branch: braincell=476, neuromorpho=476: pass
  n_bifs: braincell=237, neuromorpho=237: pass
  length: braincell=10994.445007, neuromorpho=10994.400000, rel_diff=4.09e-06 (<= 0.0001): pass
  surface: braincell=8635.016916, neuromorpho=8635.020000, rel_diff=3.57e-07 (<= 0.0001): pass
  volume: braincell=539.688557, neuromorpho=539.689000, rel_diff=8.20e-07 (<= 0.0001): pass
  pathDistance: braincell=2069.225492, neuromorpho=2069.230000, rel_diff=2.18e-06 (<= 0.0001): pass
  eucDistance: braincell=1415.655287, neuromorpho=1415.660000, rel_diff=3.33e-06 (<= 0.0001): pass
  soma_Surface: braincell=12.566371, neuromorpho=12.376000, rel_diff=1.54e-02 (<= 0.02): pass
  overall: passed
scan summary
scanned: 7
compared: 7
skipped: 0
passed: 4
registered exceptions only: 3
failed: 0
errors: 0
